In [2]:
# ============================================================
# PERSIAPAN DATA DAN EKSTRAKSI MFCC
# ============================================================

import os
import warnings
import librosa
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')

metadata_path = 'validated.tsv'
audio_folder = 'clips'
N_MFCC = 24
TARGET_SR = 16000
PRE_EMPHASIS = 0.97
WIN_LENGTH = int(TARGET_SR * 25 / 1000)
HOP_LENGTH = int(TARGET_SR * 10 / 1000)
N_FFT = 512

# Muat metadata dan pertahankan dua kelas gender yang digunakan eksperimen.
df = pd.read_csv(metadata_path, sep='\t')
df_filtered = df[
    df['gender'].isin(['male_masculine', 'female_feminine'])
].copy()

df_filtered = df_filtered[
    df_filtered['path'].map(
        lambda path: os.path.exists(os.path.join(audio_folder, path))
    )
].reset_index(drop=True)


def extract_mfcc(file_path):
    try:
        audio, sample_rate = librosa.load(
            file_path,
            sr=TARGET_SR,
            mono=True
        )
        audio = np.append(
            audio[0],
            audio[1:] - PRE_EMPHASIS * audio[:-1]
        )
        mfcc = librosa.feature.mfcc(
            y=audio,
            sr=sample_rate,
            n_mfcc=N_MFCC,
            n_fft=N_FFT,
            win_length=WIN_LENGTH,
            hop_length=HOP_LENGTH
        )
        return np.mean(mfcc, axis=1)
    except Exception as error:
        print(f'Error pada file {file_path}: {error}')
        return None


features = []
labels = []
for _, row in df_filtered.iterrows():
    feature = extract_mfcc(os.path.join(audio_folder, row['path']))
    if feature is not None:
        features.append(feature)
        labels.append(row['gender'])

X = np.asarray(features)
y = np.asarray(labels)
y_encoded = LabelEncoder().fit_transform(y)

print(f'Jumlah data valid: {len(X)}')
print(f'Dimensi fitur MFCC: {X.shape[1]}')
print('Label: male_masculine=1, female_feminine=0 (sesuai urutan LabelEncoder)')

Jumlah data valid: 21587
Dimensi fitur MFCC: 24
Label: male_masculine=1, female_feminine=0 (sesuai urutan LabelEncoder)


In [3]:
# ============================================================
# ABLATION STUDY - DATA
# ============================================================
# Konfigurasi model terbaik dari 16 eksperimen:
# MFCC = 24
# K = 3
#
# Variabel yang diubah hanya jumlah data TRAINING.
# Data TESTING dibuat tetap untuk semua skenario.
# ============================================================

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
import pandas as pd


# ============================================================
# STEP 1 - Tetapkan konfigurasi model terbaik
# ============================================================

K_BEST = 3

print("Konfigurasi model untuk Ablation Data:")
print(f"MFCC = 24")
print(f"K = {K_BEST}")


# ============================================================
# STEP 2 - Membuat TRAINING dan TESTING
# ============================================================
# Test set hanya dibuat SATU KALI dan akan digunakan
# untuk seluruh skenario ablasi.

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    stratify=y_encoded
)

print("\nPembagian data:")
print(f"Total data      : {len(X)}")
print(f"Data training   : {len(X_train_full)}")
print(f"Data testing    : {len(X_test)}")


# ============================================================
# STEP 3 - Definisi proporsi data training
# ============================================================

training_ratios = {
    "25%": 0.25,
    "50%": 0.50,
    "75%": 0.75,
    "100%": 1.00
}


# ============================================================
# STEP 4 - Menjalankan Ablation Data
# ============================================================

results_ablation = []


for label, ratio in training_ratios.items():

    print("\n" + "=" * 60)
    print(f"ABLATION DATA: {label} DATA TRAINING")
    print("=" * 60)

    # --------------------------------------------------------
    # Ambil sebagian data training
    # --------------------------------------------------------

    if ratio < 1.0:

        X_train_subset, _, y_train_subset, _ = train_test_split(
            X_train_full,
            y_train_full,
            train_size=ratio,
            stratify=y_train_full
        )

    else:

        X_train_subset = X_train_full
        y_train_subset = y_train_full


    print(f"Jumlah data training: {len(X_train_subset)}")
    print(f"Jumlah data testing : {len(X_test)}")


    # --------------------------------------------------------
    # StandardScaler
    # --------------------------------------------------------
    # Scaler FIT hanya menggunakan data training.
    # Test hanya TRANSFORM.

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train_subset)
    X_test_scaled = scaler.transform(X_test)


    # --------------------------------------------------------
    # KNN
    # --------------------------------------------------------

    knn = KNeighborsClassifier(
        n_neighbors=K_BEST
    )

    knn.fit(
        X_train_scaled,
        y_train_subset
    )


    # --------------------------------------------------------
    # Prediksi
    # --------------------------------------------------------

    y_pred = knn.predict(X_test_scaled)


    # --------------------------------------------------------
    # Evaluasi
    # --------------------------------------------------------

    cm = confusion_matrix(
        y_test,
        y_pred
    )

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    precision = precision_score(
        y_test,
        y_pred,
        average='binary'
    )

    recall = recall_score(
        y_test,
        y_pred,
        average='binary'
    )

    f1 = f1_score(
        y_test,
        y_pred,
        average='binary'
    )


    # --------------------------------------------------------
    # Simpan hasil
    # --------------------------------------------------------

    results_ablation.append({
        "Training Data": label,
        "Jumlah Training": len(X_train_subset),
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1
    })


    # --------------------------------------------------------
    # Tampilkan hasil
    # --------------------------------------------------------

    print("\nConfusion Matrix:")
    print(cm)

    print(f"\nAccuracy  : {accuracy:.4f}")
    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1-Score  : {f1:.4f}")


# ============================================================
# STEP 5 - Tabel hasil Ablation Data
# ============================================================

results_ablation_df = pd.DataFrame(
    results_ablation
)

print("\n\n" + "=" * 70)
print("HASIL ABLATION STUDY - DATA")
print("=" * 70)

print(
    results_ablation_df.to_string(
        index=False
    )
)

Konfigurasi model untuk Ablation Data:
MFCC = 24
K = 3

Pembagian data:
Total data      : 21587
Data training   : 17269
Data testing    : 4318

ABLATION DATA: 25% DATA TRAINING
Jumlah data training: 4317
Jumlah data testing : 4318

Confusion Matrix:
[[1466   55]
 [  54 2743]]

Accuracy  : 0.9748
Precision : 0.9803
Recall    : 0.9807
F1-Score  : 0.9805

ABLATION DATA: 50% DATA TRAINING
Jumlah data training: 8634
Jumlah data testing : 4318

Confusion Matrix:
[[1478   43]
 [  35 2762]]

Accuracy  : 0.9819
Precision : 0.9847
Recall    : 0.9875
F1-Score  : 0.9861

ABLATION DATA: 75% DATA TRAINING
Jumlah data training: 12951
Jumlah data testing : 4318

Confusion Matrix:
[[1482   39]
 [  29 2768]]

Accuracy  : 0.9843
Precision : 0.9861
Recall    : 0.9896
F1-Score  : 0.9879

ABLATION DATA: 100% DATA TRAINING
Jumlah data training: 17269
Jumlah data testing : 4318

Confusion Matrix:
[[1487   34]
 [  24 2773]]

Accuracy  : 0.9866
Precision : 0.9879
Recall    : 0.9914
F1-Score  : 0.9897


HASIL AB